# Step 16.3 — Batch Sampler：从连续 Token Stream 取出 `(B,T)` 的输入与右移目标

Step 16.2 已经准备好两个彼此隔离的一维 token streams：

```text
train_tokens : (N_train,)
val_tokens   : (N_val,)
```

MiniGPT 的输入接口却是 `(B,T)`。本节完成两者之间的连接：从选定 split 中抽取连续窗口，为每个位置配上 **next-token target（下一个 token 的监督目标）**，再组成 batch。

本节只处理三个核心：**block_size、shifted targets、output shapes**。沿用 Step 6.5 的 `get_batch(data, batch_size, block_size, rng)` 思路，不改变已经通过的 16.1/16.2。

材料状态：16.1 与 16.2 已通过，实际修正均为 **1/2**；本节是初始材料，实际修正 **0/2**。这里的状态不表示学员已掌握或代码已运行。母版所有 code cells 保持未执行；本轮停在 **16.3**，不进入 16.4。


## 0. 新 Work Copy 怎样取得前节的数据与定义？

新的 Notebook 可能使用新的 kernel。不能假定另一个 Notebook 中的 `token_stream`、`train_tokens` 或函数会自动出现。

本母版自包含以下依赖定义：

1. `find_project_root(start)`：从当前目录向上找到本项目，方便从 `notebooks/work/` 打开。
2. `load_cached_character_corpus(cache_path)`：复用 16.2 的本地读取与字符编码流程，返回相同的词表和 `(N,)` token stream。
3. `split_train_validation(...)`：复用 16.2 的连续 90/10 split。

这里是在新 kernel 中恢复本节所需对象，不重新教授或修改前两节。**不使用 `%run` 整份旧 Notebook，也不要求两个 Notebook 共享 kernel。**

路径有一条必须明确的规则：`CACHE_PATH` 要指向你在 16.1 已经取得的那份 `input.txt`。默认使用仓库根目录下的 `data/tinyshakespeare/input.txt`。如果 16.1 当时的 kernel 工作目录不同，旧缓存可能位于别处；在下面的路径 cell 把 `CACHE_PATH` 改为那个既有文件的绝对路径即可。

`Path.cwd()` 表示 kernel 当前目录；`parents` 是父目录序列；`resolve()` 显示绝对路径。下面的小 helper 只为找到项目位置。它不会下载或搬动语料。

缺少缓存时，读取 helper 会明确报错：之后实际练习时回到 16.1 的 work copy 完成 acquisition，或修正已有缓存路径。本节不内置网络下载。


In [ ]:
from pathlib import Path
import random
import mlx.core as mx


def find_project_root(start: Path) -> Path:
    """Find this course repository from its root or a child directory."""
    current = Path(start).resolve()
    for candidate in [current] + list(current.parents):
        project_file = candidate / 'pyproject.toml'
        previous_lesson = (
            candidate / 'notebooks' /
            '68_step16_2_train_validation_split.ipynb'
        )
        if project_file.is_file() and previous_lesson.is_file():
            return candidate
    raise FileNotFoundError(
        'Course repository not found above the kernel working directory. '
        'Open the mlx-minigpt-from-scratch folder in VS Code first.'
    )


def load_cached_character_corpus(cache_path: Path):
    """Reuse the existing local cache; never download data."""
    cache_path = Path(cache_path)
    if not cache_path.is_file():
        raise FileNotFoundError(
            f'Tiny Shakespeare cache not found at {cache_path.resolve()}. '
            'Point CACHE_PATH at the existing Step 16.1 input.txt, '
            'or complete acquisition in the Step 16.1 work copy first.'
        )

    text = cache_path.read_text(encoding='utf-8')
    characters = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(characters)}
    itos = {i: ch for i, ch in enumerate(characters)}
    encoded_ids = [stoi[ch] for ch in text]
    token_stream = mx.array(encoded_ids, dtype=mx.int32)
    return text, characters, stoi, itos, token_stream


def split_train_validation(token_stream, train_fraction: float = 0.90):
    """Reuse Step 16.2's contiguous, non-empty train/validation split."""
    if len(token_stream.shape) != 1:
        raise ValueError('token_stream must have shape (N,)')
    N = token_stream.shape[0]
    if N < 2:
        raise ValueError('token_stream must contain at least 2 tokens')
    if not 0.0 < train_fraction < 1.0:
        raise ValueError('train_fraction must satisfy 0 < train_fraction < 1')

    split_index = int(N * train_fraction)
    if split_index <= 0 or split_index >= N:
        raise ValueError('split would make train or validation data empty')

    train_tokens = token_stream[:split_index]
    val_tokens = token_stream[split_index:]
    return train_tokens, val_tokens, split_index


# Definitions only. No cache read, split, or sampling is performed here.


### 先定位既有缓存，再准备 Bootstrap Usage

下面先显示 kernel 与缓存路径。若默认路径不是 16.1 实际使用的文件，修改 `CACHE_PATH` 后再继续下一 cell；不要通过重新下载来掩盖路径不一致。

路径 cell 和 bootstrap usage cell 都是给未来 work copy 准备的代码，本轮没有执行。


In [ ]:
# USER-RUN PATH CELL — prepared only, NOT executed in the master.

PROJECT_ROOT = find_project_root(Path.cwd())
CACHE_PATH = PROJECT_ROOT / 'data' / 'tinyshakespeare' / 'input.txt'

# If the existing Step 16.1 cache is elsewhere, replace CACHE_PATH here:
# CACHE_PATH = Path('/absolute/path/to/the/existing/input.txt')

print('kernel cwd:', Path.cwd())
print('project root:', PROJECT_ROOT)
print('selected cache:', CACHE_PATH.resolve())
print('cache is a file:', CACHE_PATH.is_file())


In [ ]:
# USER-RUN BOOTSTRAP CELL — prepared only, NOT executed in the master.

text, characters, stoi, itos, token_stream = load_cached_character_corpus(
    CACHE_PATH,
)
train_tokens, val_tokens, split_index = split_train_validation(
    token_stream,
    train_fraction=0.90,
)

print('full stream:', token_stream.shape, token_stream.dtype)
print('train stream:', train_tokens.shape)
print('validation stream:', val_tokens.shape)
print('split index:', split_index)
print('vocabulary size:', len(characters))

# This usage cell only reads the existing local cache and rebuilds the split.
# It does not access the network or run a model.


## 1. Batch 与 `block_size`：一次准备多少条、每条多长？

**Batch（批次）**是一组本次一起交给模型的序列窗口。**Sampler（采样器）**决定从数据中取哪些窗口。

| 记号 / 参数 | 含义 |
|---|---|
| `N_split` | 当前选定 split 中的 token 总数 |
| `batch_size = B` | 一个 batch 有多少行，即多少个窗口 |
| `block_size = T` | 本 sampler 每行输入的固定 token 数 |
| `T_max` | MiniGPT 的最大上下文容量；之后接模型时要满足 `T <= T_max` |

例如 `B=2, block_size=8`，我们要准备两个长度为 8 的输入窗口，得到 `X.shape=(2,8)`。

本节的 sampler 每次都返回完整长度，因此 `T=block_size`。不做变长 padding，也不把整份 corpus 当成一个输入。`block_size` 是采样窗口长度，不是 Transformer 的 Block 数量。

还要为输入的最后一个位置提供“下一个 token”。所以每行真正需要从 split 读取的是：

$$
\boxed{T+1\text{ 个连续 tokens}}
$$

这里多出的一个 token 用来构造最后一个 target，不会使模型输入长度变成 T+1。


## 2. Shifted Targets：为什么 X 和 Y 只错开一个位置？

设选定 split 为一维序列 $d_0,d_1,\ldots,d_{N_{split}-1}$，当前窗口起点为 s。

对每个位置 $t=0,\ldots,T-1$：

$$
\boxed{X[t]=d_{s+t},\qquad Y[t]=d_{s+t+1}}
$$

因此：

```python
chunk = data[s : s + T + 1]  # T+1 tokens
x = chunk[:-1]              # first T tokens
y = chunk[1:]               # last T tokens
```

Python slice 的 stop 不包含在结果中。`[:-1]` 去掉最后一项，`[1:]` 去掉第一项。

### 可手算的小例子

取人为构造的 token IDs `[0,1,2,3,4,5,6,7]`，令 `T=3`。为便于看清楚，下面的两个起点是手工指定的，不是随机运行结果。

| batch 行 b | 起点 s | 取出的 T+1 个 token | X 的一行 | Y 的一行 |
|---|---:|---|---|---|
| 0 | 1 | `[1,2,3,4]` | `[1,2,3]` | `[2,3,4]` |
| 1 | 4 | `[4,5,6,7]` | `[4,5,6]` | `[5,6,7]` |

所以 `X.shape=Y.shape=(2,3)`。例如 `Y[0,2]=4` 是输入行最后一个 token 3 的下一个 token。

这里没有把首 token 接回末尾；`shift` 不是 circular roll。最后一个 target 来自真实的下一个位置。

### Target 与可见上下文的分工

X、Y 有重叠 token 是正常的。例如 `Y[b,0]` 可能正好等于 `X[b,1]`。模型在位置 0 预测时，**causal mask** 会限制它只能使用 `X[b,:1]`；位置 t 使用到 `X[b,:t+1]`。

Sampler 负责配对输入与标签；Step 12 已建立的 causal attention 负责可见性。Y 是单独交给未来 loss 的整数标签，不作为额外输入送进模型。本节只构造配对，不运行模型或实现 loss。


## 3. Random Start Index：合法起点的右端为什么是 `N_split - T`？

一个 chunk 的最后一个 target 下标为 `s+T`。为了仍落在当前 split 内：

$$
s+T\le N_{split}-1.
$$

移项得到：

$$
\boxed{0\le s\le N_{split}-T-1}
$$

等价地：

$$
\boxed{0\le s<N_{split}-T}.
$$

因此一共有 `N_split-T` 个合法起点。在刚才的 `N_split=8,T=3` 例子里，合法起点为 `0,1,2,3,4`。

### 在代码首次使用前解释 `rng.randrange`

Python 标准库的 `random.Random()` 创建一个随机数生成器对象；我们把它叫作 `rng`。`rng.randrange(0, stop)` 从 `0` 到 `stop-1` 选一个整数，**不包含 stop**。

所以本节用：

```python
start = rng.randrange(0, N_split - block_size)
```

只需要会从合法范围取起点。如何设置 seed、重复相同序列或保存随机状态属于 16.4，本节不展开，也不保证某次随机起点的具体数值。

### 两个边界直接由公式决定

| split 长度 | 能否取得 T+1 个 token？ |
|---|---|
| `N_split = T+1` | 可以，唯一合法起点为 0 |
| `N_split = T` | 不可以，缺少最后一个 target，应明确报错 |

每行独立选择起点；行之间可能重复或重叠。Batch 行号不代表文本中的先后顺序，但**每一行内部仍保持连续文本的次序**。


### Split Boundary：先选 Train 或 Validation，再采样

函数每次只接收一条已经选好的 split：

```text
get_batch(train_tokens, ...) -> training X/Y
get_batch(val_tokens, ...)   -> validation X/Y
```

不要先在完整 `token_stream` 上任意取窗口，再依据起点给它贴 train 标签；起点属于 train 并不能保证窗口最后的 target 仍属于 train。

当前接口把整个 T+1 chunk 限定在传入的 split 中，因此 X 和 Y 都遵守 16.2 的边界。

函数返回的 `starts` 是 **split-local indices（相对于当前 split 的局部下标）**。Validation 窗口在原 corpus 中的起点才是 `split_index + val_start`。Train 和 validation 即使出现相同的局部起点数值，也不代表读取了相同的原始 token positions。


## 4. 实现 `get_batch`：切片，再沿 Batch 轴 Stack

继续使用 Step 6.5 的函数接口：

```python
get_batch(data, batch_size, block_size, rng)
```

输入 data 的契约是一维整数 token stream；前面的 bootstrap 已用 `mx.int32` 构造它。本节的切片与 stack 保留这个 dtype，不把 token IDs 变成浮点特征。

三个操作先说清楚：

1. `starts` 是含 B 个起点的 Python list。
2. `xs`、`ys` 分别保存 B 个 shape 为 `(T,)` 的 MLX arrays。
3. `mx.stack(xs, axis=0)` **增加一个新轴**，把这 B 行组织成 `(B,T)`。此处不是把所有 token 连成一条 `(B*T,)` 长序列。

代码把 `axis=0` 明确写出来，使新增的第 0 轴就是 Batch 轴。

Sampler 显式拒绝非一维输入、非正整数 B/T，以及不足 T+1 的 split。它不负责重新建立词表、不切换 train/val，也不创建模型。


In [ ]:
def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


# Definition only. No batch is sampled in this cell.


### 对照数学看这段代码

`range(batch_size)` 让列表推导重复抽取 B 个起点；`_` 表示这次只需要重复次数，不使用循环编号。`xs.append(...)` 把一行加入列表。`type(value) is int` 在入口要求普通 Python 整数，避免把小数、布尔值当作窗口长度。

对每个返回行 b 与位置 t，代码实现的恰好是：

$$
X[b,t]=data[starts[b]+t],
$$

$$
Y[b,t]=data[starts[b]+t+1].
$$

`starts` 是用于观察采样位置的辅助信息：类型为 `list[int]`，长度为 B，**不是第三个 `(B,T)` 张量**。之后模型接收 X；Y 供未来的监督目标使用。


## 5. 在真实 Split 上准备 Usage：`B=2, T=8`

下面准备两行、每行 8 个 token 的小 batch。每行实际从 split 读取 9 个连续 token，用来形成 X/Y。

`rng=random.Random()` 只提供本节抽取起点所需的对象，不指定 seed，不做复现实验。两个调用分别接收 `train_tokens` 与 `val_tokens`，不会把 validation 混入训练输入。

这些 usage cells 仅供之后在 work copy 中自行执行；本轮没有读取缓存、切分或采样。


In [ ]:
# USER-RUN SAMPLING CELL — prepared only, NOT executed in the master.

B = 2
block_size = 8  # This sampler uses T == block_size.
rng = random.Random()

X_train, Y_train, train_starts = get_batch(
    train_tokens,
    batch_size=B,
    block_size=block_size,
    rng=rng,
)
X_val, Y_val, val_starts = get_batch(
    val_tokens,
    batch_size=B,
    block_size=block_size,
    rng=rng,
)

print('train X / Y shapes:', X_train.shape, Y_train.shape)
print('val X / Y shapes:', X_val.shape, Y_val.shape)
print('train X / Y dtypes:', X_train.dtype, Y_train.dtype)
print('val X / Y dtypes:', X_val.dtype, Y_val.dtype)
print('train split-local starts:', train_starts)
print('val split-local starts:', val_starts)

# Expected by construction, NOT measured here:
# all four X/Y arrays have shape (2,8) and dtype mx.int32.
# Each starts list has length 2; actual random starts are not fixed.


### Output Shapes：为什么 Y 也为 `(B,T)`？

每个输入位置都对应一个 next-token target，所以一个 batch 有 B×T 个监督位置。

| 对象 | 输出契约 | 后续用途 |
|---|---|---|
| `X_train` / `X_val` | `(B,T)`，整数 token IDs | MiniGPT 输入 |
| `Y_train` / `Y_val` | `(B,T)`，整数 target IDs | 未来 loss 的正确类别 |
| `train_starts` / `val_starts` | Python list，长度 B | 查看当前 split 内的取样位置 |

Y 此时既不是 `(B,T,V)` 的 one-hot，也不是 `(B,T,C)` 的 embedding。模型未来产生的 logits 才是 `(B,T,V)`；其中 `logits[b,t,:]` 将与单个正确类别 `Y[b,t]` 对应。

这里仅对齐接口，不实现交叉熵、optimizer 或 forward。

### 观察一行文本，不预先承诺随机内容

`array.tolist()` 把短行转成 Python 整数列表；`itos[token_id]` 查回字符；`''.join(...)` 将字符接成字符串。`repr(...)` 会让换行等空白更容易看见，沿用 16.1 的观察方式。

下面只准备输出 train 和 validation 各一行。应关注 X/Y 的一位偏移关系，具体字符串由实际采样位置决定。


In [ ]:
# USER-RUN ROW INSPECTION — prepared only, NOT executed in the master.

print('train X[0]:', repr(''.join(itos[i] for i in X_train[0].tolist())))
print('train Y[0]:', repr(''.join(itos[i] for i in Y_train[0].tolist())))
print('val X[0]  :', repr(''.join(itos[i] for i in X_val[0].tolist())))
print('val Y[0]  :', repr(''.join(itos[i] for i in Y_val[0].tolist())))

# No expected random text is supplied, and no output has been produced here.


## Work Copy 的可操作顺序与本节终点

1. 从 GitHub 更新后，安全复制本节母版为 `notebooks/work/` 下的新文件；只打开该副本，并选择项目 Python 3.12 / `.venv` kernel。
2. 先运行依赖 **definition-only** cell，得到路径、缓存读取和 split 的三个 helper；此时没有下载、读取语料或采样。
3. 运行路径 cell，确认 `CACHE_PATH` 是 16.1 的既有缓存；若默认位置不同，改成正确绝对路径，再运行该路径 cell。
4. 运行 bootstrap usage，取得 `itos`、`train_tokens`、`val_tokens`、`split_index`。它只读本地文件并复用16.2切分。
5. 阅读 T+1、shift 和合法起点推导，再运行本节 `get_batch` 的定义 cell。
6. 最后按需要运行 sampling / row-inspection usage，观察 `(B,T)`、整数 dtype、split-local starts 和每行的偏移关系。

以上是未来用户练习的操作步骤；本轮材料准备没有替用户执行，也没有填写练习答案。

到这里，数据接口已经准备为：一条选定 split → B 个 T+1 连续片段 → `X/Y:(B,T)`。

**本轮停在 Step 16.3。** 不讲授或实现 16.4 的 seeds、determinism、RNG-state resume；也不进入16.5的工业数据管线、loss、optimizer、训练或生成。

材料计数持续沿用：16.1=1/2，16.2=1/2；16.3 初版=0/2。材料准备完成不等于学员掌握或运行验证。


## 本节参考与课程衔接

- 前置：[Step 16 文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-16-tiny-shakespeare-data.md)、[Step 16.2 母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/68_step16_2_train_validation_split.ipynb)。
- 已学算法原型：[Step 6.5 sampling batches](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/21_step06_5_sampling_batches.ipynb)。本节保持同名四参数接口，并补齐真实 split 的依赖与边界。
- Python 3.12：[random / randrange](https://docs.python.org/3.12/library/random.html#random.randrange)。
- MLX API：[stack](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.stack.html)。项目依赖继续固定 `mlx==0.32.2`，本节没有升级依赖。
